# Stage 1 — Data Ingestion

Loads the two raw data sources used by the pipeline:

1. **Financial PhraseBank** (text / sentiment labels) from Hugging Face.
2. **Synthetic financial indicators** (numerical features + risk target).

Both are saved to `./artifacts/` as parquet so downstream stages can be
run independently.


In [ ]:
from common import *
from huggingface_hub import hf_hub_download

## 1.1 Text: AMZN headlines from FNSPID

MVP path: instead of calling `load_dataset("Zihan1004/FNSPID")` (which
tries to download the full ~15 GB dataset and blows past the cluster
home-directory quota), we pull just the one news CSV we need via
`hf_hub_download`, filter to AMZN in memory, and throw the rest away.

Labels are placeholder neutrals (`1`) for now so the downstream stages
stay runnable — real weak labels from forward returns come in a later
pass once price data is wired in.

In [ ]:
# Download just the FNSPID news CSV (not the full dataset).
# The filename is misspelled in the upstream repo ("exteral" -> "external");
# keep it verbatim or the download 404s.
csv_path = hf_hub_download(
    repo_id="Zihan1004/FNSPID",
    filename="Stock_news/nasdaq_exteral_data.csv",
    repo_type="dataset",
)
print(f"Downloaded to: {csv_path}")

# The CSV is a few GB; read it in chunks and keep only AMZN rows so
# we never hold the whole file in memory.
TICKER = CONFIG["company"]["ticker"] if "company" in CONFIG else "AMZN"

amzn_chunks = []
for chunk in pd.read_csv(csv_path, chunksize=500_000, low_memory=False):
    # FNSPID column names vary slightly across dumps; find them defensively.
    sym_col  = next((c for c in chunk.columns if c.lower() in {"stock_symbol", "symbol", "ticker"}), None)
    text_col = next((c for c in chunk.columns if c.lower() in {"article_title", "title", "headline"}), None)
    date_col = next((c for c in chunk.columns if c.lower() in {"date", "datetime", "publish_date"}), None)
    if sym_col is None or text_col is None or date_col is None:
        raise ValueError(f"Could not locate expected columns in FNSPID chunk: {list(chunk.columns)}")
    amzn_chunks.append(
        chunk.loc[chunk[sym_col] == TICKER, [date_col, text_col]]
             .rename(columns={date_col: "date", text_col: "text"})
    )

text_df = pd.concat(amzn_chunks, ignore_index=True)
text_df["date"] = pd.to_datetime(text_df["date"], errors="coerce").dt.tz_localize(None)
text_df = text_df.dropna(subset=["date", "text"]).drop_duplicates(subset=["date", "text"])
text_df = text_df.sort_values("date").reset_index(drop=True)

# Placeholder labels so downstream stages run. Replace with weak labels
# from forward returns once price data is wired in.
text_df["label"] = 1  # 0=neg, 1=neu, 2=pos

print(f"{TICKER} headlines: {len(text_df)}")
print(f"Date range: {text_df['date'].min().date()} to {text_df['date'].max().date()}")
print("Headlines per year:")
print(text_df["date"].dt.year.value_counts().sort_index().to_string())
text_df.head()

## 1.2 Numerical: synthetic financial indicators

In [ ]:
def generate_numerical_data(n: int = 5000, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    data = {
        "revenue_growth":     rng.normal(0.05, 0.15, n),
        "debt_to_equity":     np.abs(rng.normal(1.0, 0.8, n)),
        "current_ratio":      np.abs(rng.normal(1.5, 0.6, n)),
        "profit_margin":      rng.normal(0.10, 0.12, n),
        "log_market_cap":     rng.normal(10.0, 2.0, n),
        "beta":               np.abs(rng.normal(1.0, 0.4, n)),
        "volatility_30d":     np.abs(rng.normal(0.25, 0.15, n)),
        "pe_ratio":           np.abs(rng.normal(20.0, 12.0, n)),
        "roe":                rng.normal(0.12, 0.10, n),
        "interest_coverage":  np.abs(rng.normal(5.0, 3.0, n)),
    }
    df = pd.DataFrame(data)

    risk = (
         0.15 * (df["debt_to_equity"] / df["debt_to_equity"].max())
       + 0.15 * (1 - df["current_ratio"] / df["current_ratio"].max())
       - 0.10 * df["revenue_growth"]
       - 0.10 * df["profit_margin"]
       + 0.10 * (df["beta"] / df["beta"].max())
       + 0.15 * (df["volatility_30d"] / df["volatility_30d"].max())
       - 0.05 * (df["log_market_cap"] / df["log_market_cap"].max())
       + 0.10 * (1 / (1 + df["interest_coverage"]))
       + 0.05 * rng.normal(0, 0.1, n)
    )
    df["risk_score"] = (risk - risk.min()) / (risk.max() - risk.min())
    return df


num_df = generate_numerical_data(CONFIG["regression"]["n_synthetic_samples"])
print(f"Numerical samples: {len(num_df)}")
print(f"Risk score range: [{num_df['risk_score'].min():.3f}, {num_df['risk_score'].max():.3f}]")
num_df.describe().round(3)

## 1.3 Persist artifacts

In [ ]:
save_text_df(text_df)
save_num_df(num_df)
print("Saved:")
print(f"  {ARTIFACTS_DIR / 'text_df.parquet'}")
print(f"  {ARTIFACTS_DIR / 'num_df.parquet'}")